# Day 18 · Colab 1 — Claude Agent + FastAPI Backend + Redis Memory

**Agentic Systems Bootcamp**

In this notebook you build a small but complete **tool-using agent**:

- a **FastAPI** backend exposing an `/orders/{id}` endpoint (your *external API*),
- a **Redis** memory layer (short-term conversation history + long-term facts),
- a **Claude** agent that calls tools in a `while` loop driven by `stop_reason`.

Everything runs **inside Colab** with no external servers:
`fakeredis` stands in for Redis and FastAPI's `TestClient` calls the API in-process.
Swap either for the real thing by changing one line each (shown at the end).

> **Pattern recap (from the deck):** client tools return `stop_reason="tool_use"`; *your* code runs the
> tool and returns a `tool_result` block; you loop until `stop_reason="end_turn"`.

## Step 1 — Install dependencies & set your API key

`fakeredis` gives us a real Redis API surface in-process. `httpx` is needed by FastAPI's test client.

In [1]:
!pip install -q anthropic fastapi 'httpx<0.28' fakeredis 2>/dev/null
print('deps installed')

deps installed


The system cannot find the file specified.


In [2]:
import os, getpass
import os
from dotenv import load_dotenv

load_dotenv()
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
    # os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    

if not os.environ.get('ANTHROPIC_API_KEY'):
    # Fallback: paste it (hidden). Leave blank to run in OFFLINE mock mode.
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY (blank = offline mock): ')

LIVE = bool(os.environ.get('ANTHROPIC_API_KEY'))
MODEL = 'claude-sonnet-4-6'  # fast + cheap for a workshop; opus-4-8 for harder reasoning
print('LIVE mode' if LIVE else 'OFFLINE mock mode (no key) — agent loop will be simulated')

LIVE mode


## Step 2 — A tiny FastAPI backend (the 'external API')

This is the kind of service your agent does **not** control — it just calls it.
We expose one route, then wrap the app in a `TestClient` so calls run in-process.

In [3]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

app = FastAPI()

_ORDERS = {
    'A1001': {'id': 'A1001', 'item': 'Mechanical keyboard', 'qty': 1, 'status': 'shipped',  'total': 129.0},
    'A1002': {'id': 'A1002', 'item': 'USB-C hub',          'qty': 2, 'status': 'processing','total': 58.0},
    'A1003': {'id': 'A1003', 'item': '4K monitor',         'qty': 1, 'status': 'delivered', 'total': 410.0},
}

@app.get('/orders/{order_id}')
def get_order(order_id: str):
    o = _ORDERS.get(order_id.upper())
    if not o:
        raise HTTPException(status_code=404, detail='order not found')
    return o

client = TestClient(app)
print(client.get('/orders/A1001').json())

{'id': 'A1001', 'item': 'Mechanical keyboard', 'qty': 1, 'status': 'shipped', 'total': 129.0}


## Step 3 — A Redis memory layer

Two responsibilities, the way the deck framed memory:

- **Short-term** — the running conversation, stored as a Redis **list** per session (`hist:<session>`).
- **Long-term** — durable **facts** about the user, stored in a Redis **hash** (`facts:<session>`), with an optional TTL.

`fakeredis` implements the same commands, so this class is unchanged when you move to real Redis.

In [4]:
!pip install fakeredis

In [5]:
import json, time, fakeredis

class RedisMemory:
    def __init__(self, r, session_id: str, history_limit: int = 40):
        self.r = r
        self.sid = session_id
        self.history_limit = history_limit
        self.h_key = f'hist:{session_id}'
        self.f_key = f'facts:{session_id}'

    # ---- short-term: conversation turns ----
    def append_turn(self, role: str, content):
        self.r.rpush(self.h_key, json.dumps({'role': role, 'content': content}))
        self.r.ltrim(self.h_key, -self.history_limit, -1)  # keep only the tail

    def load_history(self):
        return [json.loads(x) for x in self.r.lrange(self.h_key, 0, -1)]

    # ---- long-term: durable facts ----
    def set_fact(self, key: str, value: str, ttl_seconds: int | None = None):
        self.r.hset(self.f_key, key, value)
        if ttl_seconds:
            self.r.expire(self.f_key, ttl_seconds)

    def get_fact(self, key: str):
        v = self.r.hget(self.f_key, key)
        return v.decode() if isinstance(v, bytes) else v

    def all_facts(self):
        return {k.decode(): v.decode() for k, v in self.r.hgetall(self.f_key).items()}

r = fakeredis.FakeStrictRedis()
mem = RedisMemory(r, session_id='demo-user')
mem.set_fact('name', 'Asha')
mem.append_turn('user', 'hello')
print('facts:', mem.all_facts())
print('history:', mem.load_history())

facts: {'name': 'Asha'}
history: [{'role': 'user', 'content': 'hello'}]


## Step 4 — Declare the tools (JSON schemas)

Three client tools. Names are **namespaced by purpose** and every field is described —
the description *is* the prompt the model reads when deciding how to call.
`order_id` uses a `pattern` so the model returns well-formed IDs.

In [6]:
TOOLS = [
    {
        'name': 'get_order',
        'description': 'Look up a customer order by its ID and return item, quantity, status and total.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'order_id': {'type': 'string', 'description': "Order ID like 'A1001'.", 'pattern': '^[Aa][0-9]{4}$'}
            },
            'required': ['order_id'],
        },
    },
    {
        'name': 'remember_fact',
        'description': 'Persist a durable fact about the user (e.g. shipping preference) for future turns.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'key':   {'type': 'string', 'description': 'Short fact key, e.g. "shipping_pref".'},
                'value': {'type': 'string', 'description': 'The fact value to store.'},
            },
            'required': ['key', 'value'],
        },
    },
    {
        'name': 'recall_fact',
        'description': 'Retrieve a previously stored fact about the user by key. Returns empty if unknown.',
        'input_schema': {
            'type': 'object',
            'properties': {'key': {'type': 'string', 'description': 'The fact key to look up.'}},
            'required': ['key'],
        },
    },
]
print(len(TOOLS), 'tools declared')

3 tools declared


## Step 5 — A dispatch map: tool name → Python function

This is the boundary between *the model's intent* and *your code*. Each function returns a
string (what the model will read back). Errors are **returned**, not raised, so the agent can recover —
that maps to `is_error: true` on the `tool_result` block in Step 6.

In [7]:
def tool_get_order(order_id: str):
    resp = client.get(f'/orders/{order_id}')
    if resp.status_code == 404:
        return {'error': f'No order {order_id} found.'}
    resp.raise_for_status()
    return resp.json()

def tool_remember_fact(key: str, value: str):
    mem.set_fact(key, value)
    return {'ok': True, 'stored': {key: value}}

def tool_recall_fact(key: str):
    v = mem.get_fact(key)
    return {'key': key, 'value': v} if v is not None else {'key': key, 'value': None}

DISPATCH = {
    'get_order': tool_get_order,
    'remember_fact': tool_remember_fact,
    'recall_fact': tool_recall_fact,
}

def run_tool(name, args):
    fn = DISPATCH.get(name)
    if fn is None:
        return {'error': f'unknown tool {name}'}, True
    try:
        out = fn(**args)
        is_err = isinstance(out, dict) and 'error' in out
        return out, is_err
    except Exception as e:
        return {'error': repr(e)}, True

print(run_tool('get_order', {'order_id': 'A1002'}))
print(run_tool('get_order', {'order_id': 'A9999'}))

({'id': 'A1002', 'item': 'USB-C hub', 'qty': 2, 'status': 'processing', 'total': 58.0}, False)
({'error': 'No order A9999 found.'}, True)


## Step 6 — The agent loop (driven by `stop_reason`)

This is the heart of client-side tool use:

1. send messages + tool definitions,
2. if `stop_reason == 'tool_use'`: run **every** `tool_use` block, append a single user message
   containing one `tool_result` per call (matched by `tool_use_id`), and loop,
3. stop at `stop_reason == 'end_turn'` and return the text.

If there's no API key we simulate one scripted tool call so the notebook still runs end-to-end.

In [8]:
import json

SYSTEM = (
    'You are an order-support assistant. Use get_order for any order question. '
    'Use remember_fact / recall_fact to keep durable user preferences across turns. '
    'Be concise.'
)

def agent_turn(user_text, max_steps=6, verbose=True):
    mem.append_turn('user', user_text)
    messages = mem.load_history()

    if not LIVE:
        # ---- offline mock: pretend the model asked for get_order once ----
        if verbose: print('… (mock) model requests get_order A1001')
        out, _ = run_tool('get_order', {'order_id': 'A1001'})
        reply = f'(mock) Order A1001 is {out.get("status", "?")}.'
        mem.append_turn('assistant', reply)
        return reply

    from anthropic import Anthropic
    clientA = Anthropic()
    for step in range(max_steps):
        resp = clientA.messages.create(
            model=MODEL, max_tokens=1024, system=SYSTEM, tools=TOOLS, messages=messages,
        )
        if resp.stop_reason == 'tool_use':
            # echo the assistant turn (text + tool_use blocks) back into the transcript
            messages.append({'role': 'assistant', 'content': [b.model_dump() for b in resp.content]})
            results = []
            for block in resp.content:
                if block.type == 'tool_use':
                    if verbose: print(f'  → tool: {block.name}({block.input})')
                    out, is_err = run_tool(block.name, block.input)
                    results.append({
                        'type': 'tool_result',
                        'tool_use_id': block.id,
                        'content': json.dumps(out),
                        'is_error': is_err,
                    })
            messages.append({'role': 'user', 'content': results})
            continue
        # end_turn
        text = ''.join(b.text for b in resp.content if b.type == 'text')
        mem.append_turn('assistant', text)
        return text
    return '(stopped: max steps reached)'

print(agent_turn('What is the status of order A1002?'))

  → tool: get_order({'order_id': 'A1002'})
Here are the details for order **A1002**:

- **Item:** USB-C Hub
- **Quantity:** 2
- **Status:** Processing
- **Total:** $58.00

Your order is currently being processed. Is there anything else I can help you with?


## Step 7 — Persistence check: memory survives across turns

Because every turn is written to Redis, a fresh `agent_turn` call still sees the history and facts.
Ask the agent to remember something, then recall it in a later turn.

In [9]:
print(agent_turn('Please remember that my shipping preference is express.'))
print('---')
print(agent_turn('What did I say my shipping preference was?'))
print('---')
print('Raw facts in Redis:', mem.all_facts())
print('History length:', len(mem.load_history()), 'turns')

  → tool: remember_fact({'key': 'shipping_pref', 'value': 'express'})
Got it! I've saved your shipping preference as **express**. I'll keep that in mind for future interactions. Is there anything else I can help you with?
---
  → tool: recall_fact({'key': 'shipping_pref'})
Your shipping preference is set to **express**. Is there anything else I can help you with?
---
Raw facts in Redis: {'name': 'Asha', 'shipping_pref': 'express'}
History length: 7 turns


## Step 8 — Multi-turn chat demo

A short scripted conversation that exercises lookup + memory together.

In [10]:
for msg in [
    'Hi, I am Asha.',
    'How much was order A1003?',
    'Remember that my budget cap is 500 dollars.',
    'Given my budget cap, was that order within it?',
]:
    print('USER:', msg)
    print('AGENT:', agent_turn(msg, verbose=False))
    print()

USER: Hi, I am Asha.
AGENT: I've saved your name, Asha! How can I assist you today?

USER: How much was order A1003?
AGENT: Order **A1003** was for a **4K Monitor** and totaled **$410.00**. It has already been **delivered**! Is there anything else I can help you with?

USER: Remember that my budget cap is 500 dollars.
AGENT: Got it, Asha! I've saved your budget cap as **$500**. Is there anything else I can help you with?

USER: Given my budget cap, was that order within it?
AGENT: Yes, order A1003 totaling **$410.00** is within your budget cap of **$500**. You still have **$90** to spare! Is there anything else I can help you with?



## Extension tasks

Pick a few — these are the assignment for this lab.

1. **Rolling summary / compaction.** When `load_history()` exceeds N turns, summarise the oldest
   turns with a cheap model call and replace them with one synthetic `assistant` summary message.
   Keep total context bounded.
2. **TTL & a `forget` tool.** Add a `forget_fact(key)` tool and give facts a TTL via `set_fact(..., ttl_seconds=...)`.
   Show that an expired fact returns `None`.
3. **Parallel lookups.** Add a `get_customer` tool and a second order endpoint, then ask a question that
   needs both — observe Claude emit **two `tool_use` blocks in one turn**. Confirm your loop answers both ids.
4. **Guarded writes / PII.** Reject `remember_fact` values that look like card numbers or emails (regex);
   return `is_error: true` with a helpful message and confirm the agent recovers.
5. **Token accounting.** Read `resp.usage` each step; print cumulative input/output tokens for a turn.
6. **Real Redis.** Replace `fakeredis.FakeStrictRedis()` with `redis.Redis.from_url(os.environ['REDIS_URL'])`
   (e.g. a free Upstash/Redis Cloud URL). The `RedisMemory` class should not change at all.

> **Stretch:** swap `TestClient` for a real deployed FastAPI URL using `httpx.Client(base_url=...)`.
> Only `tool_get_order` changes; the agent loop is untouched.

In [14]:
# Extension 1: Rolling summary / compaction
import json
from anthropic import Anthropic

COMPACTION_THRESHOLD = 6   # compact when history exceeds this many turns
KEEP_RECENT = 2            # always keep this many recent turns verbatim

def compact_history_if_needed(mem, verbose=True):
    """
    If history > COMPACTION_THRESHOLD turns, summarise the oldest turns
    with a cheap model call and replace them with one summary message.
    """
    history = mem.load_history()
    if len(history) <= COMPACTION_THRESHOLD:
        return  # nothing to do

    old_turns = history[:-KEEP_RECENT]   # everything except the last N turns
    recent_turns = history[-KEEP_RECENT:]

    # Build a prompt to summarise the old turns
    convo_text = '\n'.join(
        f"{t['role'].upper()}: {t['content'] if isinstance(t['content'], str) else json.dumps(t['content'])}"
        for t in old_turns
    )

    if LIVE:
        clientA = Anthropic()
        resp = clientA.messages.create(
            model=MODEL,
            max_tokens=300,
            messages=[{
                'role': 'user',
                'content': (
                    'Summarise the following conversation excerpt in 2-3 sentences, '
                    'preserving key facts, orders mentioned, and user preferences.\n\n'
                    + convo_text
                )
            }]
        )
        summary = resp.content[0].text
    else:
        summary = f'(mock summary of {len(old_turns)} older turns)'

    if verbose:
        print(f'[Compaction] Summarised {len(old_turns)} turns → 1 summary message')
        print(f'  Summary: {summary[:120]}...' if len(summary) > 120 else f'  Summary: {summary}')

    # Rebuild Redis history: summary message + recent verbatim turns
    summary_turn = {'role': 'assistant', 'content': f'[SUMMARY OF EARLIER CONVERSATION]: {summary}'}
    new_history = [summary_turn] + recent_turns

    # Replace history in Redis
    mem.r.delete(mem.h_key)
    for turn in new_history:
        mem.r.rpush(mem.h_key, json.dumps(turn))

# ---- Demo ----
# Populate history with several turns to trigger compaction
r_ext1 = fakeredis.FakeStrictRedis()
mem_ext1 = RedisMemory(r_ext1, session_id='ext1-user', history_limit=40)

for i, msg in enumerate(['Hi!', 'Check order A1001', 'And A1002?', 'What is A1003?',
                          'Remember I prefer express shipping', 'Thanks']):
    mem_ext1.append_turn('user', msg)
    mem_ext1.append_turn('assistant', f'(mock reply {i})')

print(f'History before compaction: {len(mem_ext1.load_history())} turns')
compact_history_if_needed(mem_ext1, verbose=True)
print(f'History after compaction:  {len(mem_ext1.load_history())} turns')
for t in mem_ext1.load_history():
    role = t['role']
    content = t['content'][:80] if isinstance(t['content'], str) else str(t['content'])[:80]
    print(f'  [{role}] {content}')

History before compaction: 12 turns
[Compaction] Summarised 10 turns → 1 summary message
  Summary: Here is a 2-3 sentence summary of the conversation:

The user inquired about three orders — **A1001**, **A1002**, and **...
History after compaction:  3 turns
  [assistant] [SUMMARY OF EARLIER CONVERSATION]: Here is a 2-3 sentence summary of the convers
  [user] Thanks
  [assistant] (mock reply 5)


In [15]:
# Extension 2: TTL & forget tool
import time, fakeredis, json

# ---- Extend RedisMemory with a delete_fact method ----
class RedisMemoryV2(RedisMemory):
    def delete_fact(self, key: str):
        """Remove a single fact by key."""
        self.r.hdel(self.f_key, key)

# ---- New tool function ----
def tool_forget_fact(key: str):
    mem_ext2.delete_fact(key)
    return {'ok': True, 'forgotten': key}

# Register into a local DISPATCH copy
DISPATCH_EXT2 = {**DISPATCH, 'forget_fact': tool_forget_fact}

# New tool schema to pass to Claude
FORGET_TOOL = {
    'name': 'forget_fact',
    'description': 'Delete a previously stored fact about the user.',
    'input_schema': {
        'type': 'object',
        'properties': {'key': {'type': 'string', 'description': 'The fact key to delete.'}},
        'required': ['key'],
    },
}
TOOLS_EXT2 = TOOLS + [FORGET_TOOL]

# ---- Demo ----
r_ext2 = fakeredis.FakeStrictRedis()
mem_ext2 = RedisMemoryV2(r_ext2, session_id='ext2-user')

# Store a fact with a short TTL (the TTL applies to the whole hash)
mem_ext2.set_fact('promo_code', 'SAVE20', ttl_seconds=2)
mem_ext2.set_fact('shipping_pref', 'express')  # no TTL

print('All facts immediately after storing:')
print(mem_ext2.all_facts())

# Test forget_fact tool
result = tool_forget_fact('shipping_pref')
print(f'\nAfter forget_fact(shipping_pref): {result}')
print('Facts now:', mem_ext2.all_facts())

# Test TTL expiry — wait for the TTL to expire
print('\nWaiting 3 seconds for promo_code TTL to expire...')
time.sleep(3)
print('promo_code after TTL:', mem_ext2.get_fact('promo_code'))   # should be None
print('All facts after TTL expiry:', mem_ext2.all_facts())         # should be empty


All facts immediately after storing:
{'promo_code': 'SAVE20', 'shipping_pref': 'express'}

After forget_fact(shipping_pref): {'ok': True, 'forgotten': 'shipping_pref'}
Facts now: {'promo_code': 'SAVE20'}

Waiting 3 seconds for promo_code TTL to expire...
promo_code after TTL: None
All facts after TTL expiry: {}


In [16]:
# Extension 3: Parallel lookups
import json, fakeredis
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

# ---- Extend the FastAPI app with a /customers endpoint ----
app_ext3 = FastAPI()

_ORDERS_EXT3 = {
    'A1001': {'id': 'A1001', 'item': 'Mechanical keyboard', 'qty': 1, 'status': 'shipped',   'total': 129.0, 'customer_id': 'C01'},
    'A1002': {'id': 'A1002', 'item': 'USB-C hub',           'qty': 2, 'status': 'processing','total': 58.0,  'customer_id': 'C02'},
    'A1003': {'id': 'A1003', 'item': '4K monitor',          'qty': 1, 'status': 'delivered', 'total': 410.0, 'customer_id': 'C01'},
}

_CUSTOMERS = {
    'C01': {'id': 'C01', 'name': 'Asha Mehta',   'email': 'asha@example.com', 'tier': 'gold'},
    'C02': {'id': 'C02', 'name': 'Raj Patel',    'email': 'raj@example.com',  'tier': 'silver'},
}

@app_ext3.get('/orders/{order_id}')
def get_order_ext3(order_id: str):
    o = _ORDERS_EXT3.get(order_id.upper())
    if not o:
        raise HTTPException(status_code=404, detail='order not found')
    return o

@app_ext3.get('/customers/{customer_id}')
def get_customer_ext3(customer_id: str):
    c = _CUSTOMERS.get(customer_id.upper())
    if not c:
        raise HTTPException(status_code=404, detail='customer not found')
    return c

client_ext3 = TestClient(app_ext3)

# ---- Tool functions ----
def tool_get_order_ext3(order_id: str):
    resp = client_ext3.get(f'/orders/{order_id}')
    if resp.status_code == 404:
        return {'error': f'No order {order_id} found.'}
    return resp.json()

def tool_get_customer(customer_id: str):
    resp = client_ext3.get(f'/customers/{customer_id}')
    if resp.status_code == 404:
        return {'error': f'No customer {customer_id} found.'}
    return resp.json()

DISPATCH_EXT3 = {
    'get_order':    tool_get_order_ext3,
    'get_customer': tool_get_customer,
    'remember_fact': tool_remember_fact,
    'recall_fact':   tool_recall_fact,
}

def run_tool_ext3(name, args):
    fn = DISPATCH_EXT3.get(name)
    if fn is None:
        return {'error': f'unknown tool {name}'}, True
    try:
        out = fn(**args)
        return out, isinstance(out, dict) and 'error' in out
    except Exception as e:
        return {'error': repr(e)}, True

TOOLS_EXT3 = [
    {
        'name': 'get_order',
        'description': 'Look up a customer order by ID. Returns item, quantity, status, total, and customer_id.',
        'input_schema': {
            'type': 'object',
            'properties': {'order_id': {'type': 'string', 'description': "Order ID like 'A1001'."}},
            'required': ['order_id'],
        },
    },
    {
        'name': 'get_customer',
        'description': 'Look up a customer profile by customer_id. Returns name, email, and loyalty tier.',
        'input_schema': {
            'type': 'object',
            'properties': {'customer_id': {'type': 'string', 'description': "Customer ID like 'C01'."}},
            'required': ['customer_id'],
        },
    },
]

# ---- Agent loop for ext3 ----
r_ext3 = fakeredis.FakeStrictRedis()
mem_ext3 = RedisMemory(r_ext3, session_id='ext3-user')

def agent_turn_ext3(user_text, max_steps=6, verbose=True):
    mem_ext3.append_turn('user', user_text)
    messages = mem_ext3.load_history()

    if not LIVE:
        # Mock: simulate two parallel tool calls in one turn
        print('… (mock) Claude emits TWO tool_use blocks in one turn:')
        order_out, _ = run_tool_ext3('get_order', {'order_id': 'A1001'})
        cust_out, _  = run_tool_ext3('get_customer', {'customer_id': order_out['customer_id']})
        print(f'  → get_order(A1001)              → {order_out}')
        print(f'  → get_customer({order_out["customer_id"]}) → {cust_out}')
        reply = f"(mock) Order A1001 ({order_out['item']}) belongs to {cust_out['name']} ({cust_out['tier']} tier)."
        mem_ext3.append_turn('assistant', reply)
        return reply

    from anthropic import Anthropic
    clientA = Anthropic()
    for step in range(max_steps):
        resp = clientA.messages.create(
            model=MODEL, max_tokens=1024,
            system='You are an order-support assistant. Use get_order then get_customer to answer questions about orders and their owners. Be concise.',
            tools=TOOLS_EXT3, messages=messages,
        )
        if resp.stop_reason == 'tool_use':
            messages.append({'role': 'assistant', 'content': [b.model_dump() for b in resp.content]})
            tool_blocks = [b for b in resp.content if b.type == 'tool_use']
            if verbose:
                print(f'  Step {step+1}: Claude emitted {len(tool_blocks)} tool_use block(s)')
            results = []
            for block in tool_blocks:
                if verbose: print(f'    → {block.name}({block.input})')
                out, is_err = run_tool_ext3(block.name, block.input)
                results.append({
                    'type': 'tool_result',
                    'tool_use_id': block.id,
                    'content': json.dumps(out),
                    'is_error': is_err,
                })
            messages.append({'role': 'user', 'content': results})
            continue
        text = ''.join(b.text for b in resp.content if b.type == 'text')
        mem_ext3.append_turn('assistant', text)
        return text
    return '(stopped: max steps reached)'

# This question requires BOTH get_order and get_customer
print(agent_turn_ext3('Tell me about order A1001 and who placed it.'))


  Step 1: Claude emitted 1 tool_use block(s)
    → get_order({'order_id': 'A1001'})
  Step 2: Claude emitted 1 tool_use block(s)
    → get_customer({'customer_id': 'C01'})
Here's the full summary:

**Order A1001**
- **Item:** Mechanical Keyboard (x1)
- **Status:** Shipped
- **Total:** $129.00

**Placed by:** Asha Mehta
- **Email:** asha@example.com
- **Loyalty Tier:** Gold


In [17]:
# Extension 5: Token accounting
import json, fakeredis

r_ext5 = fakeredis.FakeStrictRedis()
mem_ext5 = RedisMemory(r_ext5, session_id='ext5-user')

# Pricing (claude-sonnet-4-6 as of mid-2025 — update if needed)
INPUT_PRICE_PER_1M  = 3.00   # USD per 1M input tokens
OUTPUT_PRICE_PER_1M = 15.00  # USD per 1M output tokens

def agent_turn_with_token_accounting(user_text, max_steps=6, verbose=True):
    mem_ext5.append_turn('user', user_text)
    messages = mem_ext5.load_history()

    total_input  = 0
    total_output = 0

    if not LIVE:
        # Mock token accounting
        mock_input, mock_output = 320, 45
        total_input  += mock_input
        total_output += mock_output
        cost = (total_input / 1_000_000 * INPUT_PRICE_PER_1M +
                total_output / 1_000_000 * OUTPUT_PRICE_PER_1M)
        print(f'  [mock] Step 1 — input={mock_input}, output={mock_output}')
        print(f'  Totals: input={total_input}, output={total_output}, est. cost=${cost:.6f}')
        reply = '(mock) Order A1001 is shipped.'
        mem_ext5.append_turn('assistant', reply)
        return reply

    from anthropic import Anthropic
    clientA = Anthropic()

    for step in range(max_steps):
        resp = clientA.messages.create(
            model=MODEL, max_tokens=1024, system=SYSTEM, tools=TOOLS, messages=messages,
        )

        # --- Accounting ---
        step_input  = resp.usage.input_tokens
        step_output = resp.usage.output_tokens
        total_input  += step_input
        total_output += step_output

        if verbose:
            print(f'  Step {step+1}: input_tokens={step_input:,}, output_tokens={step_output:,}')

        if resp.stop_reason == 'tool_use':
            messages.append({'role': 'assistant', 'content': [b.model_dump() for b in resp.content]})
            results = []
            for block in resp.content:
                if block.type == 'tool_use':
                    if verbose: print(f'    → {block.name}({block.input})')
                    out, is_err = run_tool(block.name, block.input)
                    results.append({
                        'type': 'tool_result',
                        'tool_use_id': block.id,
                        'content': json.dumps(out),
                        'is_error': is_err,
                    })
            messages.append({'role': 'user', 'content': results})
            continue

        # end_turn — print summary
        cost = (total_input / 1_000_000 * INPUT_PRICE_PER_1M +
                total_output / 1_000_000 * OUTPUT_PRICE_PER_1M)
        print(f'  ─── Turn totals: input={total_input:,}, output={total_output:,}, est. cost=${cost:.6f}')

        text = ''.join(b.text for b in resp.content if b.type == 'text')
        mem_ext5.append_turn('assistant', text)
        return text

    return '(stopped: max steps reached)'

print('=== Token Accounting Demo ===')
reply = agent_turn_with_token_accounting('What is the status of order A1002?')
print('Reply:', reply)


=== Token Accounting Demo ===
  Step 1: input_tokens=838, output_tokens=58
    → get_order({'order_id': 'A1002'})
  Step 2: input_tokens=944, output_tokens=58
  ─── Turn totals: input=1,782, output=116, est. cost=$0.007086
Reply: Order **A1002** is currently **processing**. Here's a quick summary:

- **Item:** USB-C Hub
- **Quantity:** 2
- **Total:** $58.00

Is there anything else you'd like to know?


In [18]:
# Extension 6: Real Redis (via Upstash, Redis Cloud, or local Redis)
#
# Prerequisites:
#   pip install redis
#   Set REDIS_URL in your .env file, e.g.:
#     REDIS_URL=rediss://default:<password>@<host>:<port>   (Upstash)
#     REDIS_URL=redis://localhost:6379                      (local)
#
# Free options:
#   • Upstash  → https://upstash.com  (free tier, TLS URL)
#   • Redis Cloud → https://redis.com/try-free/

import os
from dotenv import load_dotenv
load_dotenv()

REDIS_URL = os.getenv('REDIS_URL')

if not REDIS_URL:
    print('ℹ REDIS_URL not set — falling back to fakeredis for this demo.')
    print('  To use real Redis, add REDIS_URL=<your-url> to your .env file.')
    import fakeredis
    r_real = fakeredis.FakeStrictRedis()
    backend_label = 'fakeredis (fallback)'
else:
    import redis
    r_real = redis.Redis.from_url(
        REDIS_URL,
        decode_responses=False,  # RedisMemory handles bytes → str itself
        socket_timeout=5,
    )
    r_real.ping()  # raises ConnectionError if unreachable
    backend_label = f'real Redis @ {REDIS_URL.split("@")[-1]}'

# ---- RedisMemory is UNCHANGED — just pass r_real ----
mem_real = RedisMemory(r_real, session_id='real-redis-demo')

# Clean slate for the demo
r_real.delete(mem_real.h_key, mem_real.f_key)

# Store and retrieve
mem_real.set_fact('name', 'Asha')
mem_real.set_fact('shipping_pref', 'express')
mem_real.append_turn('user', 'Hello from real Redis!')
mem_real.append_turn('assistant', 'Hi! I can remember things across sessions now.')

print(f'Backend: {backend_label}')
print('Facts :', mem_real.all_facts())
print('History:', mem_real.load_history())
print()
print('RedisMemory class was NOT modified — only the `r` argument changed. ✓')

# ---- Stretch: swap TestClient for a real FastAPI URL ----
# import httpx
# REAL_API_URL = os.getenv('ORDERS_API_URL', '')  # e.g. https://my-api.fly.dev
# if REAL_API_URL:
#     http_client = httpx.Client(base_url=REAL_API_URL, timeout=10)
#     def tool_get_order(order_id: str):
#         resp = http_client.get(f'/orders/{order_id}')
#         if resp.status_code == 404:
#             return {'error': f'No order {order_id} found.'}
#         resp.raise_for_status()
#         return resp.json()
#     print('Using real FastAPI backend at', REAL_API_URL)


ℹ REDIS_URL not set — falling back to fakeredis for this demo.
  To use real Redis, add REDIS_URL=<your-url> to your .env file.
Backend: fakeredis (fallback)
Facts : {'name': 'Asha', 'shipping_pref': 'express'}
History: [{'role': 'user', 'content': 'Hello from real Redis!'}, {'role': 'assistant', 'content': 'Hi! I can remember things across sessions now.'}]

RedisMemory class was NOT modified — only the `r` argument changed. ✓
